In [ ]:
## dEPARTMENT WISE###

In [19]:
import os
import zipfile
import pandas as pd
import warnings

# Suppress openpyxl warnings
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

# === STEP 1: UNZIP FILE ===
zip_path = "Command Area Development-CAD (DOWR).zip"
extract_dir = "Command Area Development-CAD (DOWR)"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print(f"✅ Unzipped → {extract_dir}")

# === STEP 2: FIND TENDER FOLDERS ===
base_dir = os.path.join(extract_dir, "Command Area Development-CAD (DOWR)")
tender_dirs = [
    os.path.join(base_dir, d) for d in os.listdir(base_dir)
    if os.path.isdir(os.path.join(base_dir, d))
]

# === STEP 3: FIND BOQ FILE ===
def find_boq_file(tender_folder):
    for file in os.listdir(tender_folder):
        if file.lower().startswith("boqcomparativechart") and file.endswith(".xlsx"):
            return os.path.join(tender_folder, file)
    return None

# === STEP 4: EXTRACT L1 / L2–L5 FROM 'BOQ Summary' ===
def extract_boq_summary_precise(filepath, tender_id_fallback):
    try:
        df_full = pd.read_excel(filepath, sheet_name="BOQ Summary", header=None)

        # Detect actual header row
        header_row = df_full[df_full.iloc[:, 0].astype(str).str.contains("Sheet Name", na=False)].index[0]
        df = pd.read_excel(filepath, sheet_name="BOQ Summary", header=header_row)

        # Optional: extract Tender ID from top cell
        tender_id = tender_id_fallback
        top_cell = df_full.iloc[0, 0]
        if isinstance(top_cell, str) and "Tender ID:" in top_cell:
            tender_id = top_cell.split("Tender ID:")[-1].strip()

        # Extract L1 and L2–L5 bidders
        df = df.dropna(subset=["Bidder Name", "Bid Rank"])
        df["Bid Rank"] = df["Bid Rank"].astype(str).str.strip().str.upper()

        l1_bidders = df[df["Bid Rank"] == "L1"]["Bidder Name"].tolist()
        other_bidders = df[df["Bid Rank"].isin(["L2", "L3", "L4", "L5"])]["Bidder Name"].tolist()

        return {
            "Tender ID": tender_id,
            "Contractor Name (L1)": ", ".join(l1_bidders),
            "Other Bidders (L2-L5)": ", ".join(other_bidders)
        }

    except Exception as e:
        return {
            "Tender ID": tender_id_fallback,
            "Contractor Name (L1)": f"Error: {e}",
            "Other Bidders (L2-L5)": ""
        }

# === STEP 5: LOOP & COLLECT DATA ===
results = []

for folder in tender_dirs:
    tender_id = os.path.basename(folder)
    boq_file = find_boq_file(folder)

    if boq_file:
        summary = extract_boq_summary_precise(boq_file, tender_id)
        results.append(summary)
    else:
        results.append({
            "Tender ID": tender_id,
            "Contractor Name (L1)": " No BOQ file",
            "Other Bidders (L2-L5)": ""
        })

# === STEP 6: SAVE FINAL EXCEL ===
output_df = pd.DataFrame(results)
output_file = "Command Area Development-CAD (DOWR).xlsx"
output_df.to_excel(output_file, index=False)
print(f"✅ Summary saved to → {output_file}")


✅ Unzipped → Command Area Development-CAD (DOWR)
✅ Summary saved to → Command Area Development-CAD (DOWR).xlsx


In [ ]:
##all Department wise ##

In [13]:
import os
import zipfile
import pandas as pd
import warnings

# Suppress openpyxl warnings
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

# === STEP 1: UNZIP FILE ===
zip_path = "Odisha_Govt_past_data+BOQ_download.zip"
extract_dir = "Odisha_Govt_Data_Extracted"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print(f"✅ Unzipped → {extract_dir}")

# === STEP 2: RECURSIVELY FIND ALL TENDER FOLDERS ===
tender_dirs = []
for root, dirs, files in os.walk(extract_dir):
    for d in dirs:
        tender_path = os.path.join(root, d)
        # Check if folder contains any BoQComparativeChart file
        for f in os.listdir(tender_path):
            if f.lower().startswith("boqcomparativechart") and f.lower().endswith(".xlsx"):
                tender_dirs.append(tender_path)
                break

print(f"🔍 Found {len(tender_dirs)} tender folders with BoQ files.")

# === STEP 3: Extract BOQ Summary ===
def extract_boq_summary_precise(filepath, fallback_id):
    try:
        df_full = pd.read_excel(filepath, sheet_name="BOQ Summary", header=None)

        header_row = df_full[df_full.iloc[:, 0].astype(str).str.contains("Sheet Name", na=False)].index[0]
        df = pd.read_excel(filepath, sheet_name="BOQ Summary", header=header_row)

        tender_id = fallback_id
        top_cell = df_full.iloc[0, 0]
        if isinstance(top_cell, str) and "Tender ID:" in top_cell:
            tender_id = top_cell.split("Tender ID:")[-1].strip()

        df = df.dropna(subset=["Bidder Name", "Bid Rank"])
        df["Bid Rank"] = df["Bid Rank"].astype(str).str.strip().str.upper()

        l1_bidders = df[df["Bid Rank"] == "L1"]["Bidder Name"].tolist()
        other_bidders = df[df["Bid Rank"].isin(["L2", "L3", "L4", "L5"])]["Bidder Name"].tolist()

        return {
            "Tender ID": tender_id,
            "Department": fallback_id.split(os.sep)[0],
            "Tender Folder": fallback_id.split(os.sep)[-1],
            "Contractor Name (L1)": ", ".join(l1_bidders),
            "Other Bidders (L2-L5)": ", ".join(other_bidders)
        }

    except Exception as e:
        return {
            "Tender ID": fallback_id,
            "Department": fallback_id.split(os.sep)[0],
            "Tender Folder": fallback_id.split(os.sep)[-1],
            "Contractor Name (L1)": f"Error: {e}",
            "Other Bidders (L2-L5)": ""
        }

# === STEP 4: PROCESS ALL TENDERS ===
results = []

for tender_folder in tender_dirs:
    fallback_id = os.path.relpath(tender_folder, extract_dir)
    boq_file = None
    for f in os.listdir(tender_folder):
        if f.lower().startswith("boqcomparativechart") and f.endswith(".xlsx"):
            boq_file = os.path.join(tender_folder, f)
            break

    if boq_file:
        summary = extract_boq_summary_precise(boq_file, fallback_id)
    else:
        summary = {
            "Tender ID": fallback_id,
            "Department": fallback_id.split(os.sep)[0],
            "Tender Folder": fallback_id.split(os.sep)[-1],
            "Contractor Name (L1)": "❌ No BOQ file",
            "Other Bidders (L2-L5)": ""
        }

    results.append(summary)

# === STEP 5: EXPORT TO EXCEL ===
output_df = pd.DataFrame(results)
output_file = "Odisha_Govt_All_Departments_Bidder_Summary.xlsx"
output_df.to_excel(output_file, index=False)

print(f"✅ All department summary saved to → {output_file}")


✅ Unzipped → Odisha_Govt_Data_Extracted
🔍 Found 6020 tender folders with BoQ files.
✅ All department summary saved to → Odisha_Govt_All_Departments_Bidder_Summary.xlsx


In [14]:

import os
import zipfile
import pandas as pd
import warnings

# Suppress openpyxl warnings
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

# === STEP 1: UNZIP FILE ===
zip_path = "Odisha_Govt_past_data+BOQ_download.zip"
extract_dir = "Odisha_Govt_Data_Extracted"

if not os.path.exists(extract_dir):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
    print(f"✅ Unzipped → {extract_dir}")
else:
    print(f"ℹ️ Already extracted → {extract_dir}")

# === STEP 2: RECURSIVELY FIND ALL TENDER FOLDERS ===
tender_dirs = []
for root, dirs, files in os.walk(extract_dir):
    for d in dirs:
        tender_path = os.path.join(root, d)
        for f in os.listdir(tender_path):
            if f.lower().startswith("boqcomparativechart") and f.lower().endswith(".xlsx"):
                tender_dirs.append(tender_path)
                break

print(f"🔍 Found {len(tender_dirs)} tender folders with BoQ files.")

# === STEP 3: EXTRACT BOQ SUMMARY ===
def extract_boq_summary_precise(filepath, fallback_id):
    try:
        # Try reading the sheet (name might be slightly off)
        all_sheets = pd.ExcelFile(filepath).sheet_names
        boq_sheet = next((s for s in all_sheets if "boq summary" in s.lower()), None)

        if boq_sheet is None:
            raise ValueError("No sheet named 'BOQ Summary' found.")

        df_full = pd.read_excel(filepath, sheet_name=boq_sheet, header=None)

        header_row = df_full[df_full.iloc[:, 0].astype(str).str.contains("Sheet Name", na=False)].index
        header_row = header_row[0] if not header_row.empty else 0

        df = pd.read_excel(filepath, sheet_name=boq_sheet, header=header_row)

        # Attempt to extract Tender ID from first cell
        tender_id = fallback_id
        top_cell = str(df_full.iloc[0, 0]) if not pd.isna(df_full.iloc[0, 0]) else ""
        if "Tender ID:" in top_cell:
            tender_id = top_cell.split("Tender ID:")[-1].strip()

        # Clean Bidder Rank
        df = df.dropna(subset=["Bidder Name", "Bid Rank"])
        df["Bid Rank"] = df["Bid Rank"].astype(str).str.strip().str.upper()

        l1_bidders = df[df["Bid Rank"] == "L1"]["Bidder Name"].astype(str).tolist()
        other_bidders = df[df["Bid Rank"].isin(["L2", "L3", "L4", "L5"])]["Bidder Name"].astype(str).tolist()

        return {
            "Tender ID": tender_id,
            "Department": fallback_id.split(os.sep)[0],
            "Tender Folder": fallback_id.split(os.sep)[-1],
            "Contractor Name (L1)": ", ".join(l1_bidders),
            "Other Bidders (L2-L5)": ", ".join(other_bidders)
        }

    except Exception as e:
        return {
            "Tender ID": fallback_id,
            "Department": fallback_id.split(os.sep)[0],
            "Tender Folder": fallback_id.split(os.sep)[-1],
            "Contractor Name (L1)": f"⚠️ Error: {e}",
            "Other Bidders (L2-L5)": ""
        }

# === STEP 4: PROCESS ALL TENDER FOLDERS ===
results = []
for tender_folder in tender_dirs:
    fallback_id = os.path.relpath(tender_folder, extract_dir)
    boq_file = next((f for f in os.listdir(tender_folder)
                     if f.lower().startswith("boqcomparativechart") and f.lower().endswith(".xlsx")), None)

    if boq_file:
        full_path = os.path.join(tender_folder, boq_file)
        summary = extract_boq_summary_precise(full_path, fallback_id)
    else:
        summary = {
            "Tender ID": fallback_id,
            "Department": fallback_id.split(os.sep)[0],
            "Tender Folder": fallback_id.split(os.sep)[-1],
            "Contractor Name (L1)": " No BOQ file",
            "Other Bidders (L2-L5)": ""
        }

    results.append(summary)

# === STEP 5: EXPORT FINAL SUMMARY ===
output_df = pd.DataFrame(results)
output_file = "Odisha_Govt_All_Bidder_Summary.xlsx"
output_df.to_excel(output_file, index=False)

print(f"\n✅ All department bidder summary saved to → {output_file}")


ℹ️ Already extracted → Odisha_Govt_Data_Extracted
🔍 Found 6020 tender folders with BoQ files.

✅ All department bidder summary saved to → Odisha_Govt_All_Bidder_Summary.xlsx


In [1]:
import os
import zipfile
import pandas as pd
import warnings

# Suppress openpyxl warnings
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

# === STEP 1: UNZIP FILE ===
zip_path = "TAMRALIPTA CO-OPERATIVE SPINNING MILLS LTD.zip"
extract_dir = "TAMRALIPTA CO-OPERATIVE SPINNING MILLS LTDa"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print(f"✅ Unzipped → {extract_dir}")

# === STEP 2: RECURSIVELY FIND ALL TENDER FOLDERS ===
tender_dirs = []
for root, dirs, files in os.walk(extract_dir):
    for d in dirs:
        tender_path = os.path.join(root, d)
        for f in os.listdir(tender_path):
            if f.lower().startswith("boqcomparativechart") and f.lower().endswith(".xlsx"):
                tender_dirs.append(tender_path)
                break

print(f"🔍 Found {len(tender_dirs)} tender folders with BoQ files.")

# === STEP 3: Extract BOQ Summary ===
def extract_boq_summary_precise(filepath, fallback_id):
    try:
        df_full = pd.read_excel(filepath, sheet_name="BOQ Summary", header=None)

        header_row = df_full[df_full.iloc[:, 0].astype(str).str.contains("Sheet Name", na=False)].index[0]
        df = pd.read_excel(filepath, sheet_name="BOQ Summary", header=header_row)

        tender_id = fallback_id
        top_cell = df_full.iloc[0, 0]
        if isinstance(top_cell, str) and "Tender ID:" in top_cell:
            tender_id = top_cell.split("Tender ID:")[-1].strip()

        df = df.dropna(subset=["Bidder Name", "Bid Rank"])
        df["Bid Rank"] = df["Bid Rank"].astype(str).str.strip().str.upper()

        l1_bidders = df[df["Bid Rank"] == "L1"]["Bidder Name"].tolist()
        other_bidders = df[df["Bid Rank"].isin(["L2", "L3", "L4", "L5"])]["Bidder Name"].tolist()

        return {
            "Tender ID": tender_id,
            "Department": fallback_id.split(os.sep)[0],
            "Tender Folder": fallback_id.split(os.sep)[-1],
            "Contractor Name (L1)": ", ".join(l1_bidders),
            "Other Bidders (L2-L5)": ", ".join(other_bidders),
            "Status": "✅ BOQ file processed"
        }

    except Exception as e:
        return {
            "Tender ID": fallback_id,
            "Department": fallback_id.split(os.sep)[0],
            "Tender Folder": fallback_id.split(os.sep)[-1],
            "Contractor Name (L1)": f"Error: {e}",
            "Other Bidders (L2-L5)": "",
            "Status": "⚠️ Error reading BOQ file"
        }

# === STEP 4: PROCESS ALL FOLDERS ===
results = []
processed_dirs = set(tender_dirs)

# Get all department/tender subfolders
all_subfolders = []
for root, dirs, files in os.walk(extract_dir):
    for d in dirs:
        all_subfolders.append(os.path.join(root, d))

for folder in all_subfolders:
    fallback_id = os.path.relpath(folder, extract_dir)

    # Look for BOQ file
    boq_file = None
    for f in os.listdir(folder):
        if f.lower().startswith("boqcomparativechart") and f.lower().endswith(".xlsx"):
            boq_file = os.path.join(folder, f)
            break

    if boq_file:
        summary = extract_boq_summary_precise(boq_file, fallback_id)
    else:
        summary = {
            "Tender ID": fallback_id,
            "Department": fallback_id.split(os.sep)[0],
            "Tender Folder": fallback_id.split(os.sep)[-1],
            "Contractor Name (L1)": "No BOQ file",
            "Other Bidders (L2-L5)": "",
            "Status": "❌ No BOQ file found"
        }

    results.append(summary)

# === STEP 5: EXPORT TO EXCEL ===
output_df = pd.DataFrame(results)
output_file = "TAMRALIPTA_past_3years_Bidder_Summary.xlsx"
output_df.to_excel(output_file, index=False)

print(f"✅ All department summary saved to → {output_file}")


✅ Unzipped → TAMRALIPTA CO-OPERATIVE SPINNING MILLS LTDa
🔍 Found 36 tender folders with BoQ files.
✅ All department summary saved to → TAMRALIPTA_past_3years_Bidder_Summary.xlsx
